# Session 7: システム同定
# Session 7: System Identification

## 目標 / Objective

実機データから Allan 分散、センサノイズ特性、慣性モーメントを同定する。

Identify Allan variance, sensor noise characteristics, and moments of inertia from real data.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| Allan 分散 | ジャイロのノイズ特性を評価 |
| センサノイズモデル | ホワイトノイズ vs バイアス不安定性 |
| 慣性モーメント同定 | ステップ応答からの推定 |
| sf sysid コマンド | 自動同定ツール |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from stampfly_edu import load_sample_data

print("Ready! / 準備完了！")

## 1. センサノイズデータの取得 / Acquiring Sensor Noise Data

### 実機の場合

StampFly を水平な面に静置し、60秒間のセンサデータを記録：

```bash
sf sysid noise --duration 60
```

### サンプルデータ

In [ ]:
# Load static noise data (60s)
# 静止ノイズデータを読み込む (60秒)
noise_data = load_sample_data("static_noise")

print(f"Samples: {len(noise_data)}")
print(f"Duration: {noise_data['time'].iloc[-1]:.1f} s")
print(f"Sample rate: {1.0 / (noise_data['time'].iloc[1] - noise_data['time'].iloc[0]):.0f} Hz")
print(f"\nColumns: {list(noise_data.columns)}")
noise_data.head()

## 2. ジャイロノイズの可視化 / Gyro Noise Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, col, label in zip(axes, ['p', 'q', 'r'],
                           ['Gyro X (Roll)', 'Gyro Y (Pitch)', 'Gyro Z (Yaw)']):
    ax.plot(noise_data['time'], noise_data[col], linewidth=0.3, alpha=0.7)
    ax.set_ylabel(f'{label}\n(rad/s)')
    ax.grid(True, alpha=0.3)
    
    std = noise_data[col].std()
    mean = noise_data[col].mean()
    ax.axhline(y=mean, color='r', linestyle='--', linewidth=0.5)
    ax.text(0.98, 0.95, f'std={std:.2e} rad/s\nmean={mean:.2e}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

axes[-1].set_xlabel('Time (s) / 時間')
fig.suptitle('Gyroscope Noise / ジャイロノイズ', fontsize=14)
fig.tight_layout()
plt.show()

## 3. Allan 分散 / Allan Variance

### 理論 / Theory

Allan 分散は時系列のノイズ特性を **時間スケール (τ)** で分析します：

$$\sigma^2(\tau) = \frac{1}{2(N-1)} \sum_{i=1}^{N-1} (\bar{y}_{i+1} - \bar{y}_i)^2$$

Log-log プロットの傾きからノイズの種類がわかります：

| 傾き | ノイズ種類 | 意味 |
|------|-----------|------|
| -1/2 | ホワイトノイズ (ARW) | ランダムウォーク |
| 0 | バイアス不安定性 | 最小値の位置 |
| +1/2 | レートランダムウォーク | 長時間ドリフト |

In [ ]:
def allan_variance(data, dt, max_cluster=None):
    """Compute Allan variance.

    Allan 分散を計算する。

    Args:
        data: 1D time series / 1次元時系列
        dt: Sample period (s) / サンプル周期
        max_cluster: Max cluster size / 最大クラスタサイズ

    Returns:
        tau: Averaging times / 平均化時間
        adev: Allan deviation / Allan 偏差
    """
    n = len(data)
    if max_cluster is None:
        max_cluster = n // 4
    
    # Cluster sizes (powers of 2)
    # クラスタサイズ（2の累乗）
    m_values = []
    m = 1
    while m <= max_cluster:
        m_values.append(m)
        m = int(m * 1.5) + 1
    
    tau = []
    adev = []
    
    for m in m_values:
        n_clusters = n // m
        if n_clusters < 2:
            break
        
        # Compute cluster averages
        # クラスタ平均を計算
        clusters = data[:n_clusters * m].reshape(n_clusters, m).mean(axis=1)
        
        # Allan variance
        diff = np.diff(clusters)
        avar = 0.5 * np.mean(diff**2)
        
        tau.append(m * dt)
        adev.append(np.sqrt(avar))
    
    return np.array(tau), np.array(adev)

In [ ]:
# Compute Allan variance for each gyro axis
# 各ジャイロ軸の Allan 分散を計算
dt = noise_data['time'].iloc[1] - noise_data['time'].iloc[0]

fig, ax = plt.subplots(figsize=(10, 7))

for col, label, color in [('p', 'Gyro X', 'r'), ('q', 'Gyro Y', 'g'), ('r', 'Gyro Z', 'b')]:
    tau, adev = allan_variance(noise_data[col].values, dt)
    ax.loglog(tau, adev, f'{color}-', linewidth=1.5, label=label)

# Reference slopes
# 参照傾斜
tau_ref = np.logspace(-2, 1, 100)
ax.loglog(tau_ref, 0.01 / np.sqrt(tau_ref), 'k--', linewidth=0.5,
          alpha=0.5, label='slope -1/2 (white noise)')

ax.set_xlabel('Averaging time τ (s) / 平均化時間')
ax.set_ylabel('Allan deviation σ(τ) (rad/s) / Allan 偏差')
ax.set_title('Allan Deviation Plot / Allan 偏差プロット')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Extract ARW (at τ=1s)
# ARW の抽出（τ=1s で）
for col, label in [('p', 'Gyro X'), ('q', 'Gyro Y'), ('r', 'Gyro Z')]:
    tau, adev = allan_variance(noise_data[col].values, dt)
    # Interpolate at τ=1s
    arw_idx = np.argmin(np.abs(tau - 1.0))
    arw = adev[arw_idx]
    bi_idx = np.argmin(adev)
    bi = adev[bi_idx]
    bi_tau = tau[bi_idx]
    print(f"{label}: ARW ≈ {arw:.2e} rad/s, Bias instability ≈ {bi:.2e} rad/s at τ={bi_tau:.1f}s")

## 4. 加速度計ノイズ / Accelerometer Noise

In [ ]:
# Accelerometer noise histogram
# 加速度計ノイズのヒストグラム
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, col, label in zip(axes, ['ax', 'ay', 'az'],
                           ['Accel X', 'Accel Y', 'Accel Z']):
    ax.hist(noise_data[col], bins=50, density=True, alpha=0.7, edgecolor='black')
    ax.set_xlabel(f'{label} (m/s²)')
    ax.set_ylabel('Density / 密度')
    
    mean = noise_data[col].mean()
    std = noise_data[col].std()
    ax.set_title(f'{label}\nμ={mean:.4f}, σ={std:.4f}')

fig.suptitle('Accelerometer Noise Distribution / 加速度計ノイズ分布', fontsize=14)
fig.tight_layout()
plt.show()

## 5. 実機での同定 / Real Hardware Identification

### sf sysid コマンド

```bash
# Sensor noise identification
# センサノイズ同定
sf sysid noise --duration 60

# Moment of inertia identification
# 慣性モーメント同定
sf sysid inertia --axis roll
sf sysid inertia --axis pitch
```

## 6. 考察課題 / Discussion Questions

1. **Allan 分散の読み方**: 最小値が小さいほど良いセンサと言えるか？
2. **ESKF パラメータ**: Allan 分散から得られた値を ESKF の Q 行列にどう反映させるか？
3. **温度の影響**: センサノイズ特性は温度によってどう変化するか？

---

1. **Reading Allan Variance**: Does a smaller minimum mean a better sensor?
2. **ESKF Parameters**: How do Allan variance results map to ESKF Q matrix?
3. **Temperature Effects**: How do sensor noise characteristics change with temperature?

In [ ]:
# Your experiments here / ここで実験してみてください
